# Tibetan Research SDK Starter v2

This notebook exercises the SDK end to end and makes the recent SDK improvements observable in Jupyter.

## Expected behavior
- Repeated `sdk.embed_sentences(...)` calls with the same model-load settings should reuse one cached `TextEmbedder` instance.
- `sdk.pairwise(...)` should reuse the SDK's existing segmenter instead of resolving a new one internally.
- Model loading controls should flow through the SDK into the embedder: `torch_dtype`, `device_map`, and `load_in_8bit`.


In [1]:
import sys
from pathlib import Path
from pprint import pprint
from time import perf_counter
from unittest.mock import patch

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "tibetan_pipeline").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise RuntimeError("Could not find project root containing tibetan_pipeline/ and requirements.txt")

project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tibetan_pipeline import TibetanResearchSDK

print(f"project_root={project_root}")


/opt/homebrew/Caskroom/miniconda/base/envs/embedding-tibetan-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0405 23:59:48.292000 77555 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


project_root=/Users/ten-jampa/Documents/personal_projects/.worktrees/embedding-model-for-tibetan/sdk-embedding-loading


In [2]:
sdk = TibetanResearchSDK(
    engine="botok_ours",
    source_format="unicode",
    model_id="buddhist-nlp/gemma-2-mitra-e",
    device="cpu",
    batch_size=1,
    embedding_progress="off",
)

sdk


In [3]:
text_a = "བོད་ཀྱི་རི་ཁྲོད་ན་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐོས་པ་ཡོད། ཉི་མ་ཤར་བའི་དུས་སུ་རི་རྩེ་དང་གངས་རི་དག་ལ་གསལ་པོར་འོད་འཕྲོ་བ་མཐོང་ཐུབ། མི་ཚོས་ས་ཆ་དེར་ཞི་བའི་སེམས་དང་དགའ་བ་ཆེན་པོ་མྱོང་གི་ཡོད། རང་བཞིན་གྱི་མཛེས་སྡུག་དེས་མི་ཚོའི་སེམས་ལ་བདེ་བ་སྐྱེད་པ་ཡིན།"
text_b = "སློབ་གྲྭའི་ནང་དུ་སློབ་མ་ཚོས་ཉིན་རེར་ཤེས་ཡོན་གསར་པ་མང་པོ་སློབ་ཀྱི་ཡོད། དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་བཤད་བྱས་ཏེ་རིག་པ་འཕེལ་བར་བྱེད། སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས་པས་སློབ་སྦྱོང་ལ་དགའ་བ་སྐྱེས། དེ་ལས་སློབ་མ་ཚོའི་མི་ཚེའི་མདུན་ལམ་ཡང་དེ་བས་གསལ་བོ་འགྱོ་བ་ཡིན།"

seg_a = sdk.segment_text(text_a)
seg_b = sdk.segment_text(text_b)

display(seg_a.to_dataframe())
display(seg_b.to_dataframe())


[INFO] Downloading general dialect pack ...
[INFO] Download completed!


/opt/homebrew/Caskroom/miniconda/base/envs/embedding-tibetan-env/lib/python3.11/site-packages/botok/textunits/bostring.py:82: UserWarning: Beware of unexpected results: input string contains the non-expanded char "དྷ", found in "".
  warn(
/opt/homebrew/Caskroom/miniconda/base/envs/embedding-tibetan-env/lib/python3.11/site-packages/botok/textunits/bostring.py:82: UserWarning: Beware of unexpected results: input string contains the non-expanded char "ྡྷ", found in "ྡྷ་པཱ་".
  warn(
/opt/homebrew/Caskroom/miniconda/base/envs/embedding-tibetan-env/lib/python3.11/site-packages/botok/textunits/bostring.py:82: UserWarning: Beware of unexpected results: input string contains the non-expanded char "བྷ", found in "".
  warn(
/opt/homebrew/Caskroom/miniconda/base/envs/embedding-tibetan-env/lib/python3.11/site-packages/botok/textunits/bostring.py:82: UserWarning: Beware of unexpected results: input string contains the non-expanded char "ྦྷ", found in "".
  warn(
/opt/homebrew/Caskroom/miniconda/base/e

,segment_index,start,end,segment_text
0,0,0,66,བོད་ཀྱི་རི་ཁྲོད་ན་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉི...
1,1,66,136,ཉི་མ་ཤར་བའི་དུས་སུ་རི་རྩེ་དང་གངས་རི་དག་ལ་གསལ་པ...
2,2,136,192,མི་ཚོས་ས་ཆ་དེར་ཞི་བའི་སེམས་དང་དགའ་བ་ཆེན་པོ་མྱོ...
3,3,192,251,རང་བཞིན་གྱི་མཛེས་སྡུག་དེས་མི་ཚོའི་སེམས་ལ་བདེ་བ...


,segment_index,start,end,segment_text
0,0,0,70,སློབ་གྲྭའི་ནང་དུ་སློབ་མ་ཚོས་ཉིན་རེར་ཤེས་ཡོན་གས...
1,1,70,147,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...
2,2,147,223,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...
3,3,223,284,དེ་ལས་སློབ་མ་ཚོའི་མི་ཚེའི་མདུན་ལམ་ཡང་དེ་བས་གསལ...


## Check 1: Embedder caching

Expected result:
- `len(sdk._embedders)` stays at `1` after multiple `embed_sentences(...)` calls with the same model-load settings.
- The embedder object id stays the same even if `batch_size` or `embedding_progress` changes.
- The second call may still do inference work, but it should not require a fresh model load.


In [4]:
t0 = perf_counter()
embedding_view_a = sdk.embed_sentences(seg_a.segments)
t1 = perf_counter()

embedder_ids_after_first = [id(embedder) for embedder in sdk._embedders.values()]
cache_size_after_first = len(sdk._embedders)

t2 = perf_counter()
embedding_view_b = sdk.embed_sentences(
    seg_b.segments,
    batch_size=2,
    embedding_progress="batch",
)
t3 = perf_counter()

embedder_ids_after_second = [id(embedder) for embedder in sdk._embedders.values()]
cache_size_after_second = len(sdk._embedders)

assert cache_size_after_first == 1
assert cache_size_after_second == 1
assert embedder_ids_after_first == embedder_ids_after_second

print({
    "cache_size_after_first": cache_size_after_first,
    "cache_size_after_second": cache_size_after_second,
    "embedder_ids_after_first": embedder_ids_after_first,
    "embedder_ids_after_second": embedder_ids_after_second,
    "first_call_seconds": round(t1 - t0, 3),
    "second_call_seconds": round(t3 - t2, 3),
})

display(embedding_view_a.to_dataframe().head())
display(embedding_view_b.to_dataframe().head())


The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 464/464 [00:00<00:00, 1745.49it/s, Materializing param=model.norm.weight]                                


[embed] backend=gemma-last-token device=cpu total_sentences=4 batch_size=2 batches=2
[embed] batch 1/2 sentences 1-2/4
[embed] batch 2/2 sentences 3-4/4
{'cache_size_after_first': 1, 'cache_size_after_second': 1, 'embedder_ids_after_first': [4759082192], 'embedder_ids_after_second': [4759082192], 'first_call_seconds': 215.135, 'second_call_seconds': 105.674}


,sentence_index,sentence_text,vector_norm
0,0,བོད་ཀྱི་རི་ཁྲོད་ན་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉི...,1.0
1,1,ཉི་མ་ཤར་བའི་དུས་སུ་རི་རྩེ་དང་གངས་རི་དག་ལ་གསལ་པ...,1.0
2,2,མི་ཚོས་ས་ཆ་དེར་ཞི་བའི་སེམས་དང་དགའ་བ་ཆེན་པོ་མྱོ...,1.0
3,3,རང་བཞིན་གྱི་མཛེས་སྡུག་དེས་མི་ཚོའི་སེམས་ལ་བདེ་བ...,1.0


,sentence_index,sentence_text,vector_norm
0,0,སློབ་གྲྭའི་ནང་དུ་སློབ་མ་ཚོས་ཉིན་རེར་ཤེས་ཡོན་གས...,1.0
1,1,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...,1.0
2,2,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...,1.0
3,3,དེ་ལས་སློབ་མ་ཚོའི་མི་ཚེའི་མདུན་ལམ་ཡང་དེ་བས་གསལ...,1.0


## Check 2: `pairwise()` should reuse the SDK segmenter

This cell patches `tibetan_pipeline.sdk.resolve_segmenter` to raise if it is called after SDK construction. If the fix worked, `sdk.pairwise(...)` still succeeds because it reuses `sdk._segmenter`.


In [5]:
with patch("tibetan_pipeline.sdk.resolve_segmenter", side_effect=AssertionError("pairwise() should not construct a new segmenter")):
    pairwise_view = sdk.pairwise(text_a, text_b, top_k=5, batch_size=2)

display(pairwise_view.topk_dataframe())
print({
    "pairwise_segments_a": len(pairwise_view.segments_a),
    "pairwise_segments_b": len(pairwise_view.segments_b),
    "similarity_shape": pairwise_view.similarity_matrix.shape,
    "cached_embedders": len(sdk._embedders),
})


,rank,score,i,j,sentence_a,sentence_b
0,1,0.448630,2,2,མི་ཚོས་ས་ཆ་དེར་ཞི་བའི་སེམས་དང་དགའ་བ་ཆེན་པོ་མྱོ...,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...
1,2,0.406273,3,2,རང་བཞིན་གྱི་མཛེས་སྡུག་དེས་མི་ཚོའི་སེམས་ལ་བདེ་བ...,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...
2,3,0.391034,1,1,ཉི་མ་ཤར་བའི་དུས་སུ་རི་རྩེ་དང་གངས་རི་དག་ལ་གསལ་པ...,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...
3,4,0.374388,3,1,རང་བཞིན་གྱི་མཛེས་སྡུག་དེས་མི་ཚོའི་སེམས་ལ་བདེ་བ...,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...
4,5,0.373295,1,2,ཉི་མ་ཤར་བའི་དུས་སུ་རི་རྩེ་དང་གངས་རི་དག་ལ་གསལ་པ...,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...


{'pairwise_segments_a': 4, 'pairwise_segments_b': 4, 'similarity_shape': (4, 4), 'cached_embedders': 1}


In [8]:
saraswati_praise_text = """
༄༅། །སྒྲ་དབྱངས་ལྷ་མོ་དབྱངས་ཅན་མ་ལ་བསྟོད་པ།
 
ཨོཾ་བདེ་ལེགས་སུ་གྱུར་ཅིག
 

ཆུ་འཛིན་དཀར་པོའི་གློག་ཕྲེང་དྲ་བ་ཅན། །
མཁའ་ཡི་མཛེས་བྱེད་འདྲ་བའི་ཡིད་འཕྲོག་མ། །
དྲི་ཟའི་ན་ཆུང་དབུས་ན་འཇོ་སྒེག་མཁན། །
རིང་ནས་བརྩེ་བའི་ལྷ་མོ་ད་ཚུར་བྱོན། །
 

པད་མའི་བཞིན་ལ་གཡོ་ལྡན་བུང་བའི་མིག །
མཐོན་མཐིང་རལ་པའི་རྩེ་ན་འོད་དཀར་ཅན། །
རོལ་སྒེག་གར་གྱིས་འགྱིང་བའི་དབྱངས་ཅན་མ། །
ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །
 

རོལ་རྩེད་གར་གྱི་ཉམས་ལྡན་རི་དྭགས་མིག །
མིག་གིས་བལྟ་བས་མི་ངོམ་ཡིད་འཕྲོག་མ། །
མ་ལྟར་བརྩེ་བ་ཁྱེད་ཀྱིས་བདག་གི་ངག །
ངག་དབང་ལྷ་མོ་ཉིད་དང་མཚུངས་པར་མཛོད། །
 

སྟོན་ཟླ་རྒྱས་པའི་དཔལ་ལས་ལྷག་པར་མཛེས། །
ཚངས་དབྱངས་སྙན་པའི་གདངས་ཀྱང་ཟིལ་གྱིས་གནོན། །
ཟབ་ཡངས་རྒྱ་མཚོའི་འཇིང་ལྟར་དཔག་དཀའ་བ། །
དབྱངས་ཅན་ལྷ་མོའི་སྐུ་གསུང་ཐུགས་ལ་འདུད། །
"""

In [9]:
foundation_of_all_good_qualities_text = """
༄༅། །ཡོན་ཏན་གཞིར་གྱུར་མ་བཞུགས་སོ། །
 

ཡོན་ཏན་ཀུན་གྱི་གཞིར་གྱུར་དྲིན་ཅན་རྗེ། །
ཚུལ་བཞིན་བསྟེན་པ་ལམ་གྱི་རྩ་བ་རུ། །
ལེགས་པར་མཐོང་ནས་འབད་པ་དུ་མ་ཡིས། །
གུས་པ་ཆེན་པོས་བསྟེན་པར་བྱིན་གྱིས་རློབས། །
 

ལན་ཅིག་རྙེད་པའི་དལ་བའི་རྟེན་བཟང་འདི། །
ཤིན་ཏུ་རྙེད་དཀའ་དོན་ཆེ་ཤེས་གྱུར་ནས། །
ཉིན་མཚན་ཀུན་ཏུ་སྙིང་པོ་ལེན་པའི་བློ། །
རྒྱུན་ཆད་མེད་པར་སྐྱེ་བར་བྱིན་གྱིས་རློབས། །
 

ལུས་སྲོག་གཡོ་བ་ཆུ་ཡི་ཆུ་བུར་བཞིན། །
མྱུར་དུ་འཇིག་པའི་འཆི་བ་དྲན་པ་དང༌། །
ཤི་བའི་རྗེས་སུ་ལུས་དང་གྲིབ་མ་བཞིན། །
དཀར་ནག་ལས་འབྲས་ཕྱི་བཞིན་འབྲང་བ་ལ། །
 

ངེས་པ་བརྟན་པོ་རྙེད་ནས་ཉེས་པའི་ཚོགས། །
ཕྲ་ཞིང་ཕྲ་བ་རྣམས་ཀྱང་སྤོང་བ་དང༌། །
དགེ་ཚོགས་མཐའ་དག་སྒྲུབ་པར་བྱེད་པ་ལ། །
རྟག་ཏུ་བག་དང་ལྡན་པར་བྱིན་གྱིས་རློབས། །
 

སྤྱད་པས་མི་ངོམས་སྡུག་བསྔལ་ཀུན་གྱི་སྒོ། །
ཡིད་བརྟན་མི་རུང་སྲིད་པའི་ཕུན་ཚོགས་ཀྱི། །
ཉེས་དམིགས་རིག་ནས་ཐར་བའི་བདེ་བ་ལ། །
དོན་གཉེར་ཆེན་པོ་སྐྱེ་བར་བྱིན་གྱིས་རློབས། །
 

རྣམ་དག་བསམ་པ་དེ་ཡིས་དྲངས་པ་ཡི། །
དྲན་དང་ཤེས་བཞིན་བག་ཡོད་ཆེན་པོ་ཡིས། །
བསྟན་པའི་རྩ་བ་སོ་སོར་ཐར་བ་ལ། །
སྒྲུབ་པ་སྙིང་པོར་བྱེད་པར་བྱིན་གྱིས་རློབས། །
 

རང་ཉིད་སྲིད་མཚོར་ལྷུང་བ་ཇི་བཞིན་དུ། །
མར་གྱུར་འགྲོ་བ་ཀུན་ཀྱང་དེ་འདྲ་བར། །
མཐོང་ནས་འགྲོ་བ་སྒྲོལ་བའི་ཁུར་འཁྱེར་བའི། །
བྱང་ཆུབ་སེམས་མཆོག་འབྱོངས་པར་བྱིན་གྱིས་རློབས། །
 

སེམས་ཙམ་བསྐྱེད་ཀྱང་ཚུལ་ཁྲིམས་རྣམ་གསུམ་ལ། །
གོམས་པ་མེད་ན་བྱང་ཆུབ་མི་འགྲུབ་པར། །
ལེགས་པར་མཐོང་ནས་རྒྱལ་སྲས་སྡོམ་པ་ལ། །
བརྩོན་པ་དྲག་པོས་སློབ་པར་བྱིན་གྱིས་རློབས། །
 

ལོག་པའི་ཡུལ་ལ་གཡེངས་པ་ཞི་བྱེད་ཅིང༌། །
ཡང་དག་དོན་ལ་ཚུལ་བཞིན་དཔྱོད་པ་ཡིས། །
ཞི་གནས་ལྷག་མཐོང་ཟུང་དུ་འབྲེལ་བའི་ལམ། །
མྱུར་དུ་རྒྱུད་ལ་སྐྱེ་བར་བྱིན་གྱིས་རློབས། །
 

ཐུན་མོང་ལམ་སྦྱང་སྣོད་དུ་གྱུར་པ་ན། །
ཐེག་པ་ཀུན་གྱི་མཆོག་གྱུར་རྡོ་རྗེ་ཐེག།
སྐལ་བཟང་སྐྱེ་བོའི་འཇུག་ངོགས་དམ་པ་དེར། །
བདེ་བླག་ཉིད་དུ་འཇུག་པར་བྱིན་གྱིས་རློབས། །
 

དེ་ཚེ་དངོས་གྲུབ་རྣམ་གཉིས་སྒྲུབ་པའི་གཞི། །
རྣམ་དག་དམ་ཚིག་སྡོམ་པར་གསུངས་པ་ལ། །
བཅོས་མ་མིན་པའི་ངེས་པ་རྙེད་གྱུར་ནས། །
སྲོག་དང་བསྡོས་ཏེ་སྲུང་བར་བྱིན་གྱིས་རློབས། །
 

དེ་ནས་རྒྱུད་སྡེ་སྙིང་པོ་རིམ་གཉིས་ཀྱི། །
གནད་རྣམས་ཇི་བཞིན་རྟོགས་ནས་བརྩོན་པ་ཡིས། །
ཐུན་བཞིའི་རྣལ་འབྱོར་སྤྱོད་ལས་མི་གཡེལ་བར། །
དམ་པའི་གསུང་བཞིན་སྒྲུབ་པར་བྱིན་གྱིས་རློབས། །
 

དེ་ལྟར་ལམ་བཟང་སྟོན་པའི་བཤེས་གཉེན་དང༌། །
ཚུལ་བཞིན་སྒྲུབ་པའི་གྲོགས་རྣམས་ཞབས་བརྟན་ཅིང༌། །
ཕྱི་དང་ནང་གི་བར་དུ་གཅོད་པའི་ཚོགས། །
ཉེ་བར་ཞི་བར་བྱིན་གྱིས་བརླབ་ཏུ་གསོལ། །
 

སྐྱེ་བ་ཀུན་ཏུ་ཡང་དག་བླ་མ་དང་། །
འབྲལ་མེད་ཆོས་ཀྱི་དཔལ་ལ་ལོངས་སྤྱོད་ཅིང་། །
ས་དང་ལམ་གྱི་ཡོན་ཏན་རབ་རྫོགས་ནས། །
རྡོ་རྗེ་འཆང་གི་གོ་འཕང་མྱུར་ཐོབ་ཤོག །
 

ཅེས་པ་འདི་ནི་རྗེ་ཙོང་ཁ་པ་བློ་བཟང་གྲགས་པས་མཛད་པའོ། །
"""

In [10]:
detailed_pairwise_view = sdk.pairwise(saraswati_praise_text, foundation_of_all_good_qualities_text, top_k=10)
detailed_pairwise_view.topk_dataframe()


,rank,score,i,j,sentence_a,sentence_b
0,1,0.587507,8,44,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,དམ་པའི་གསུང་བཞིན་སྒྲུབ་པར་བྱིན་གྱིས་རློབས། །
1,2,0.530747,8,21,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,སྒྲུབ་པ་སྙིང་པོར་བྱེད་པར་བྱིན་གྱིས་རློབས། །
2,3,0.513949,8,29,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,བརྩོན་པ་དྲག་པོས་སློབ་པར་བྱིན་གྱིས་རློབས། །
3,4,0.513827,8,17,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,དོན་གཉེར་ཆེན་པོ་སྐྱེ་བར་བྱིན་གྱིས་རློབས། །
4,5,0.511090,11,44,མ་ལྟར་བརྩེ་བ་ཁྱེད་ཀྱིས་བདག་གི་ངག །,དམ་པའི་གསུང་བཞིན་སྒྲུབ་པར་བྱིན་གྱིས་རློབས། །
5,6,0.510514,8,48,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,ཉེ་བར་ཞི་བར་བྱིན་གྱིས་བརླབ་ཏུ་གསོལ། །
6,7,0.505490,8,33,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,མྱུར་དུ་རྒྱུད་ལ་སྐྱེ་བར་བྱིན་གྱིས་རློབས། །
7,8,0.494472,8,37,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,བདེ་བླག་ཉིད་དུ་འཇུག་པར་བྱིན་གྱིས་རློབས། །
8,9,0.491337,8,7,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,རྒྱུན་ཆད་མེད་པར་སྐྱེ་བར་བྱིན་གྱིས་རློབས། །
9,10,0.491304,8,25,ད་དུང་བདག་ལ་ངག་གི་དངོས་གྲུབ་སྩོལ། །,བྱང་ཆུབ་སེམས་མཆོག་འབྱོངས་པར་བྱིན་གྱིས་རློབས། །


## Long-text pairwise from precomputed embeddings

Use `pairwise_from_embedding_views(...)` when you already have embeddings in memory. This avoids the expensive re-embedding step inside `pairwise()`.


In [17]:
import sys
import importlib

modules_to_reload = [
    "tibetan_pipeline.embeddings",
    "tibetan_pipeline.pairwise",
    "tibetan_pipeline.pipeline",
    "tibetan_pipeline.sdk",
    "tibetan_pipeline",
]

for name in modules_to_reload:
    if name in sys.modules:
        importlib.reload(sys.modules[name])

from tibetan_pipeline import TibetanResearchSDK

sdk_reloaded = TibetanResearchSDK(
      engine="botok_ours",
      model_id="buddhist-nlp/gemma-2-mitra-e",
      device="mps",
      batch_size=1,
)

In [18]:

t2 = perf_counter()
detailed_pairwise_cached_view = sdk_reloaded.pairwise_from_embedding_views(
    embedding_view_a,
    embedding_view_b,
    top_k=10,
)
t3 = perf_counter()

print({
    "saraswati_segments": len(seg_a.segments),
    "foundation_segments": len(seg_b.segments),
    "embedding_seconds": round(t1 - t0, 3),
    "pairwise_from_embeddings_seconds": round(t3 - t2, 3),
    "similarity_shape": detailed_pairwise_cached_view.similarity_matrix.shape,
})

detailed_pairwise_cached_view.topk_dataframe()


{'saraswati_segments': 4, 'foundation_segments': 4, 'embedding_seconds': 215.135, 'pairwise_from_embeddings_seconds': 0.011, 'similarity_shape': (4, 4)}


,rank,score,i,j,sentence_a,sentence_b
0,1,0.471028,2,2,མི་ཚོས་ས་ཆ་དེར་ཞི་བའི་སེམས་དང་དགའ་བ་ཆེན་པོ་མྱོ...,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...
1,2,0.426306,3,2,རང་བཞིན་གྱི་མཛེས་སྡུག་དེས་མི་ཚོའི་སེམས་ལ་བདེ་བ...,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...
2,3,0.373088,1,2,ཉི་མ་ཤར་བའི་དུས་སུ་རི་རྩེ་དང་གངས་རི་དག་ལ་གསལ་པ...,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...
3,4,0.371984,3,1,རང་བཞིན་གྱི་མཛེས་སྡུག་དེས་མི་ཚོའི་སེམས་ལ་བདེ་བ...,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...
4,5,0.364294,1,1,ཉི་མ་ཤར་བའི་དུས་སུ་རི་རྩེ་དང་གངས་རི་དག་ལ་གསལ་པ...,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...
5,6,0.363550,0,2,བོད་ཀྱི་རི་ཁྲོད་ན་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉི...,སློབ་མ་ཚོས་མཉམ་དུ་ལས་ཀ་བྱས་པ་དང་གྲོས་བསྡུར་བྱས...
6,7,0.358841,2,1,མི་ཚོས་ས་ཆ་དེར་ཞི་བའི་སེམས་དང་དགའ་བ་ཆེན་པོ་མྱོ...,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...
7,8,0.304387,0,1,བོད་ཀྱི་རི་ཁྲོད་ན་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉི...,དགེ་རྒན་ཚོས་སློབ་མ་རྣམས་ལ་དྲི་བ་འདྲི་བ་དང་གསལ་...
8,9,0.295192,2,0,མི་ཚོས་ས་ཆ་དེར་ཞི་བའི་སེམས་དང་དགའ་བ་ཆེན་པོ་མྱོ...,སློབ་གྲྭའི་ནང་དུ་སློབ་མ་ཚོས་ཉིན་རེར་ཤེས་ཡོན་གས...
9,10,0.269828,0,0,བོད་ཀྱི་རི་ཁྲོད་ན་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉི...,སློབ་གྲྭའི་ནང་དུ་སློབ་མ་ཚོས་ཉིན་རེར་ཤེས་ཡོན་གས...


## Check 3: Model loading controls are available

This check verifies that the SDK forwards model-loading options into the cached embedder config.

Worth knowing:
- `torch_dtype` controls weight precision at load time.
- `device_map` lets Transformers place model parts automatically or explicitly.
- `load_in_8bit=True` is mainly useful on CUDA setups with bitsandbytes support.


In [11]:
sdk_controlled = TibetanResearchSDK(
    engine="botok_ours",
    source_format="unicode",
    model_id="buddhist-nlp/gemma-2-mitra-e",
    device="cpu",
    batch_size=1,
    torch_dtype="float32",
)

controlled_embedder = sdk_controlled._get_embedder(
    model_id=sdk_controlled.model_id,
    batch_size=sdk_controlled.batch_size,
    device=sdk_controlled.device,
    embedding_progress=sdk_controlled.embedding_progress,
    torch_dtype=sdk_controlled.torch_dtype,
    device_map=sdk_controlled.device_map,
    load_in_8bit=sdk_controlled.load_in_8bit,
)

pprint({
    "torch_dtype": controlled_embedder.torch_dtype,
    "device_map": controlled_embedder.device_map,
    "load_in_8bit": controlled_embedder.load_in_8bit,
})


{'device_map': None, 'load_in_8bit': False, 'torch_dtype': 'float32'}


## MPS SDK

In [12]:
mps_sdk = TibetanResearchSDK(
      engine="botok_ours",
      model_id="buddhist-nlp/gemma-2-mitra-e",
      device="mps",
      batch_size=1,
  )

In [13]:
with patch("tibetan_pipeline.sdk.resolve_segmenter", side_effect=AssertionError("pairwise() should not construct a new segmenter")):
    mps_pairwise_view = mps_sdk.pairwise(text_a, text_b, top_k=5, batch_size=2)

display(mps_pairwise_view.topk_dataframe())
print({
    "pairwise_segments_a": len(mps_pairwise_view.segments_a),
    "pairwise_segments_b": len(mps_pairwise_view.segments_b),
    "similarity_shape": mps_pairwise_view.similarity_matrix.shape,
    "cached_embedders": len(mps_sdk._embedders),
})


Loading weights: 100%|██████████| 464/464 [00:00<00:00, 1611.17it/s, Materializing param=model.norm.weight]                                


RuntimeError: MPS out of memory while loading embedding model. Rerun with device='cpu' (CLI: --device cpu).

## Optional CUDA-oriented configuration example

Do not run this cell on CPU-only setups. It is here as a copy-paste example for a CUDA machine where you want lower VRAM usage.


In [7]:
# sdk_cuda_8bit = TibetanResearchSDK(
#     engine="botok_ours",
#     source_format="unicode",
#     model_id="buddhist-nlp/gemma-2-mitra-e",
#     device="cuda",
#     batch_size=4,
#     device_map="auto",
#     load_in_8bit=True,
# )
# sdk_cuda_8bit.embed_sentences(seg_a.segments[:2])
